# SoleSense — Dataset Exploration
**Dataset:** StepUP-P150 (StepUP-P150: High-Resolution Plantar Pressure, 150 subjects)  
**Run from:** project root `f:/Dataset/SoleSense/`

This notebook explores the dataset and feature distributions used in the SoleSense pipeline.  
Plots are saved to `docs/figures/`.

In [ ]:
import sys, os
# Ensure imports work from notebook
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

FIGURES_DIR = os.path.join(project_root, 'docs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

FEATURES_CSV = os.path.join(project_root, 'data', 'features', 'features.csv')
DATASET_ROOT = os.path.join(project_root, '..', 'FRDR_dataset_1280_download_590_202609031103', 'py')

print('Project root:', project_root)
print('Features CSV:', FEATURES_CSV)
print('Exists:', os.path.exists(FEATURES_CSV))

In [ ]:
# Load feature table
df = pd.read_csv(FEATURES_CSV)
print(f'Feature table: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Subjects: {df["subject_id"].nunique()} | '
      f'Footwear: {df["footwear"].unique().tolist()} | '
      f'Trials: {df["trial"].unique().tolist()}')
df.head(3)

## 1. Raw Pressure Signal — Single Footstep

In [ ]:
from src.data_loader import load_walking_trial

trial_df = load_walking_trial('001', 'BF', 'W1', include_pressure_arrays=True,
                               dataset_root=DATASET_ROOT)

# Pick first valid step
valid = trial_df[trial_df['exclude'] == 0].reset_index(drop=True)
arr = valid.loc[0, 'pressure_array']  # shape (101, 75, 40)

# Total force per frame
total_force = arr.sum(axis=(1,2))
frames = np.arange(len(total_force)) / 100.0  # convert to seconds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(frames, total_force, color='#2196F3', linewidth=2)
axes[0].fill_between(frames, total_force, alpha=0.2, color='#2196F3')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Total Force (kPa sum)')
axes[0].set_title('Raw Force-Time Curve — Single Footstep')
axes[0].grid(True, alpha=0.3)

# Peak frame pressure map
peak_idx = total_force.argmax()
im = axes[1].imshow(arr[peak_idx], cmap='hot', origin='upper', aspect='auto')
axes[1].set_title(f'Peak Pressure Frame (t={peak_idx/100:.2f}s)')
axes[1].set_xlabel('Medial → Lateral (px)')
axes[1].set_ylabel('Toe (0) → Heel (74)')
plt.colorbar(im, ax=axes[1], label='Pressure (kPa)')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '01_raw_pressure_signal.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_raw_pressure_signal.png')

## 2. Pressure Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Peak pressure distribution
for fw, color in zip(['BF','ST','P1','P2'], ['#2196F3','#4CAF50','#FF9800','#E91E63']):
    subset = df[df['footwear'] == fw]['press_max_kpa']
    axes[0].hist(subset, bins=40, alpha=0.5, label=fw, color=color, density=True)
axes[0].set_xlabel('Peak Pressure (kPa)')
axes[0].set_ylabel('Density')
axes[0].set_title('Peak Pressure Distribution by Footwear')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PTI distribution
for side, color in zip(['Left','Right'], ['#2196F3','#E91E63']):
    subset = df[df['side'] == side]['pti_total_kpa_s']
    axes[1].hist(subset, bins=40, alpha=0.6, label=side, color=color, density=True)
axes[1].set_xlabel('Pressure-Time Integral (kPa·s)')
axes[1].set_ylabel('Density')
axes[1].set_title('Pressure-Time Integral: Left vs Right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Stance time distribution
axes[2].hist(df['stance_time_s'], bins=50, color='#9C27B0', alpha=0.7, density=True)
axes[2].axvline(df['stance_time_s'].mean(), color='red', linestyle='--', label=f'Mean={df["stance_time_s"].mean():.2f}s')
axes[2].set_xlabel('Stance Time (s)')
axes[2].set_ylabel('Density')
axes[2].set_title('Stance Duration Distribution')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '02_pressure_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 02_pressure_distributions.png')

## 3. Pressure Heatmap — Mean Plantar Pressure Map

In [ ]:
from src.data_loader import load_walking_trial

# Load one subject's BF W1 trial and average peak frames
trial_df2 = load_walking_trial('001', 'BF', 'W1', include_pressure_arrays=True,
                                dataset_root=DATASET_ROOT)
valid2 = trial_df2[trial_df2['exclude'] == 0].reset_index(drop=True)

left_maps, right_maps = [], []
for _, row in valid2.iterrows():
    arr = row['pressure_array']
    if arr is None: continue
    # mean across all frames
    mean_frame = arr.mean(axis=0)  # (75, 40)
    if row['side'] == 'Left':
        left_maps.append(mean_frame)
    else:
        right_maps.append(mean_frame)

left_avg = np.mean(left_maps, axis=0) if left_maps else np.zeros((75, 40))
right_avg = np.mean(right_maps, axis=0) if right_maps else np.zeros((75, 40))

vmax = max(left_avg.max(), right_avg.max())

fig, axes = plt.subplots(1, 2, figsize=(9, 8))
for ax, data, title in zip(axes, [left_avg, right_avg], ['Left Foot', 'Right Foot']):
    im = ax.imshow(data, cmap='plasma', origin='upper', aspect='auto',
                   vmin=0, vmax=vmax)
    ax.set_title(f'{title}\nMean Plantar Pressure Map', fontsize=13)
    ax.set_xlabel('Medial → Lateral')
    ax.set_ylabel('Toe (0) → Heel (74)')
    ax.axhline(15, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(35, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(55, color='white', linewidth=0.8, linestyle='--', alpha=0.6)
    for y, label in zip([7, 25, 45, 65], ['Toe','Forefoot','Midfoot','Rearfoot']):
        ax.text(41, y, label, color='white', fontsize=8, va='center')
    plt.colorbar(im, ax=ax, label='Mean Pressure (kPa)')

plt.suptitle('Subject 001 — BF W1 — Mean Plantar Pressure', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_plantar_pressure_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 03_plantar_pressure_heatmap.png')

## 4. Regional Loading Distribution

In [ ]:
regions = ['load_frac_toe', 'load_frac_forefoot', 'load_frac_midfoot', 'load_frac_rearfoot']
labels = ['Toe', 'Forefoot', 'Midfoot', 'Rearfoot']
colors = ['#FF5722', '#FF9800', '#4CAF50', '#2196F3']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by region
data_by_region = [df[r].dropna().values * 100 for r in regions]
bp = axes[0].boxplot(data_by_region, labels=labels, patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Load Fraction (%)')
axes[0].set_title('Regional Loading Distribution (all subjects)')
axes[0].grid(True, alpha=0.3, axis='y')

# Mean by footwear
fw_means = df.groupby('footwear')[regions].mean() * 100
x = np.arange(len(fw_means))
width = 0.2
for i, (col, label, color) in enumerate(zip(regions, labels, colors)):
    axes[1].bar(x + i*width, fw_means[col], width, label=label, color=color, alpha=0.8)
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(fw_means.index)
axes[1].set_ylabel('Mean Load Fraction (%)')
axes[1].set_title('Regional Loading by Footwear Condition')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_regional_loading.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_regional_loading.png')

## 5. Gait Timing Distributions

In [ ]:
# One row per trial (deduplicate)
trial_df_gait = df.groupby(['subject_id','footwear','trial']).first().reset_index()
trial_df_gait = trial_df_gait.dropna(subset=['cadence_steps_per_min','gait_symmetry_index'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Cadence by trial
trial_order = ['W1','W2','W3','W4']
trial_labels = ['W1\nPreferred','W2\nSlow-Stop','W3\nSlower','W4\nFaster']
cad_data = [trial_df_gait[trial_df_gait['trial']==t]['cadence_steps_per_min'].dropna().values
            for t in trial_order]
bp2 = axes[0].boxplot(cad_data, labels=trial_labels, patch_artist=True)
for patch, color in zip(bp2['boxes'], ['#2196F3','#4CAF50','#FF9800','#E91E63']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].set_ylabel('Cadence (steps/min)')
axes[0].set_title('Cadence by Walking Speed')
axes[0].grid(True, alpha=0.3, axis='y')

# Stance time by trial
st_data = [trial_df_gait[trial_df_gait['trial']==t]['stance_mean_s'].dropna().values
           for t in trial_order]
bp3 = axes[1].boxplot(st_data, labels=trial_labels, patch_artist=True)
for patch, color in zip(bp3['boxes'], ['#2196F3','#4CAF50','#FF9800','#E91E63']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Mean Stance Time (s)')
axes[1].set_title('Stance Duration by Walking Speed')
axes[1].grid(True, alpha=0.3, axis='y')

# Gait symmetry index
axes[2].hist(trial_df_gait['gait_symmetry_index'].dropna() * 100, bins=30,
             color='#9C27B0', alpha=0.7, density=True)
axes[2].axvline(10, color='orange', linestyle='--', label='10% threshold')
axes[2].set_xlabel('Gait Symmetry Index (%)')
axes[2].set_ylabel('Density')
axes[2].set_title('Gait Symmetry Index Distribution')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_gait_timing.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 05_gait_timing.png')

## 6. Left ↔ Right Comparison

In [ ]:
left_df = df[df['side']=='Left']
right_df = df[df['side']=='Right']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# PTI scatter left vs right (per-trial means)
trial_pti = df.groupby(['subject_id','footwear','trial','side'])['pti_total_kpa_s'].mean().unstack('side').dropna()
if 'Left' in trial_pti and 'Right' in trial_pti:
    axes[0].scatter(trial_pti['Left'], trial_pti['Right'], alpha=0.4, s=15, color='#2196F3')
    lims = [min(trial_pti.min().min(), trial_pti.min().min()),
            max(trial_pti.max().max(), trial_pti.max().max())]
    axes[0].plot(lims, lims, 'r--', linewidth=1, label='Perfect symmetry')
    axes[0].set_xlabel('Left PTI (kPa·s)')
    axes[0].set_ylabel('Right PTI (kPa·s)')
    axes[0].set_title('Left vs Right PTI (per trial)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Asymmetry distribution
asym_col = df.groupby(['subject_id','footwear','trial'])['asym_pti_total_kpa_s'].first().dropna()
axes[1].hist(asym_col, bins=30, color='#FF5722', alpha=0.7, density=True)
axes[1].axvline(10, color='orange', linestyle='--', label='10% moderate')
axes[1].axvline(20, color='red', linestyle='--', label='20% high')
axes[1].set_xlabel('PTI Asymmetry (%)')
axes[1].set_ylabel('Density')
axes[1].set_title('Loading Asymmetry Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Regional asymmetry
region_asym = ['asym_load_toe','asym_load_forefoot','asym_load_midfoot','asym_load_rearfoot']
r_labels = ['Toe','Forefoot','Midfoot','Rearfoot']
asym_by_region = df.groupby(['subject_id','footwear','trial'])[region_asym].first().dropna()
bp4 = axes[2].boxplot([asym_by_region[c] for c in region_asym], labels=r_labels,
                       patch_artist=True, notch=False)
for patch, color in zip(bp4['boxes'], ['#FF5722','#FF9800','#4CAF50','#2196F3']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[2].set_ylabel('Asymmetry (%)')
axes[2].set_title('Regional Loading Asymmetry')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '06_left_right_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 06_left_right_comparison.png')

## 7. CoP Trajectory

In [ ]:
from src.pressure_features import compute_cop

# Plot CoP trajectories for first 6 valid steps
trial_df3 = load_walking_trial('001', 'BF', 'W1', include_pressure_arrays=True,
                                dataset_root=DATASET_ROOT)
valid3 = trial_df3[trial_df3['exclude'] == 0].reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

_ROWS_NP = np.arange(75, dtype=float)
_COLS_NP = np.arange(40, dtype=float)

for ax_i, (_, row) in enumerate(valid3.head(6).iterrows()):
    arr = row['pressure_array']
    if arr is None: continue
    per_frame_total = arr.sum(axis=(1, 2))
    active_mask = per_frame_total > 0
    if active_mask.sum() < 2: continue
    active_arr = arr[active_mask]
    active_tot = per_frame_total[active_mask]
    cop_row = (active_arr * _ROWS_NP[:, None]).sum(axis=(1,2)) / active_tot * 0.5
    cop_col = (active_arr * _COLS_NP[None, :]).sum(axis=(1,2)) / active_tot * 0.5

    # Background: mean pressure
    mean_img = arr.mean(axis=0)
    axes[ax_i].imshow(mean_img, cmap='Greys', origin='upper', aspect='auto',
                       extent=[0, 20, 37.5, 0])
    sc = axes[ax_i].scatter(cop_col, cop_row, c=np.arange(len(cop_row)),
                             cmap='plasma', s=20, zorder=5)
    axes[ax_i].plot(cop_col, cop_row, 'w-', linewidth=0.8, alpha=0.7, zorder=4)
    axes[ax_i].plot(cop_col[0], cop_row[0], 'go', markersize=8, label='Strike', zorder=6)
    axes[ax_i].plot(cop_col[-1], cop_row[-1], 'r^', markersize=8, label='Toe-off', zorder=6)
    axes[ax_i].set_title(f'Step {int(row["footstep_id"])} — {row["side"]}')
    axes[ax_i].set_xlabel('Lat (cm)')
    axes[ax_i].set_ylabel('Toe→Heel (cm)')
    if ax_i == 0: axes[ax_i].legend(fontsize=8)

plt.suptitle('Center of Pressure Trajectories — Subject 001 BF W1', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_cop_trajectories.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_cop_trajectories.png')

## 8. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
missing_report = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_report = missing_report[missing_report['missing'] > 0].sort_values('pct', ascending=False)

if len(missing_report) == 0:
    print('No missing values in feature table.')
else:
    print(f'{len(missing_report)} columns with missing values:')
    print(missing_report.to_string())
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_report)*0.3)))
    missing_report['pct'].plot(kind='barh', ax=ax, color='#FF5722', alpha=0.8)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Analysis')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, '08_missing_values.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 9. Feature Correlation Matrix (selected features)

In [ ]:
key_feats = [
    'press_max_kpa', 'pti_total_kpa_s', 'contact_area_cm2',
    'load_frac_forefoot', 'load_frac_rearfoot', 'load_frac_midfoot',
    'cop_path_length_cm', 'cop_ap_range_cm', 'stance_time_s',
    'asym_pti_total_kpa_s', 'gait_symmetry_index', 'cadence_steps_per_min',
]
key_feats = [f for f in key_feats if f in df.columns]
corr = df[key_feats].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(key_feats)))
ax.set_yticks(range(len(key_feats)))
ax.set_xticklabels([f.replace('_',' ') for f in key_feats], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([f.replace('_',' ') for f in key_feats], fontsize=9)
for i in range(len(key_feats)):
    for j in range(len(key_feats)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Feature Correlation Matrix — Key SoleSense Features')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '09_feature_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 09_feature_correlation.png')

## 10. Risk Score Distribution

In [ ]:
import sys, os
sys.path.insert(0, project_root)
from src.risk_engine import score_features_df, score_trial_summary

scored_df = score_features_df(df)
trial_summary = score_trial_summary(scored_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Risk score histogram
colors_map = {'NORMAL': '#4CAF50', 'MONITOR': '#FF9800', 'ALERT': '#F44336'}
for level, color in colors_map.items():
    subset = scored_df[scored_df['risk_level'] == level]['risk_score']
    if len(subset): axes[0].hist(subset, bins=25, alpha=0.7, label=level, color=color, density=True)
axes[0].axvline(30, color='orange', linestyle='--', linewidth=1.5)
axes[0].axvline(60, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Density')
axes[0].set_title('SoleSense Risk Indicator Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Risk level pie
level_counts = trial_summary['risk_level'].value_counts()
colors_pie = [colors_map.get(l, 'grey') for l in level_counts.index]
axes[1].pie(level_counts.values, labels=level_counts.index, colors=colors_pie,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Trial Risk Level Distribution')

# Risk score by footwear
fw_order = ['BF', 'ST', 'P1', 'P2']
fw_data = [trial_summary[trial_summary['footwear']==fw]['risk_score_mean'].dropna().values
           for fw in fw_order]
bp5 = axes[2].boxplot(fw_data, labels=fw_order, patch_artist=True)
for patch, color in zip(bp5['boxes'], ['#2196F3','#4CAF50','#FF9800','#E91E63']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[2].set_ylabel('Mean Risk Score (per trial)')
axes[2].set_title('Risk Score by Footwear Condition')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('SoleSense Risk Indicator — EXPERIMENTAL, not clinically validated', 
             fontsize=10, color='grey', style='italic')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '10_risk_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 10_risk_score_distribution.png')

In [ ]:
print('=== Exploration complete ===')
print(f'Figures saved to: {FIGURES_DIR}')
import os
figs = [f for f in os.listdir(FIGURES_DIR) if f.endswith('.png')]
for f in sorted(figs):
    print(' ', f)